# 01 — 主解析: RF基準とGaussian Processカーネル比較

このノートブックが本プロジェクトの中心です。受領した3つのCSVを使い、(1) R `randomForest` + residual PLS5の再現、(2) 同じ固定10-foldで複数のGPRカーネルを比較、(3) 最良GPRがRFをどこで改善し、どこで失敗するかを確認します。

重要: 標準化、角度変換、分散ゼロ列除去、`X_proc` のPCAは各訓練foldだけでfitします。

In [ ]:
# GitHubから開いたColabはファイルを自動取得しないため、最初にcloneします。
from pathlib import Path
import os, subprocess, sys

REPO_URL = 'https://github.com/futoshi-futami/Chemistory.git'
REPO_REF = os.environ.get('CHEMISTORY_REF', 'main')
FALLBACK_REF = 'agent/rbf-kernel-comparison'  # PR #2。mainへmerge後はfallback不要
cwd = Path.cwd()
if (cwd / 'pyproject.toml').exists():
    PROJECT_ROOT = cwd
elif (Path('/content/Chemistory') / 'pyproject.toml').exists():
    PROJECT_ROOT = Path('/content/Chemistory')
elif 'google.colab' in sys.modules:
    PROJECT_ROOT = Path('/content/Chemistory')
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    raise FileNotFoundError('Chemistory project rootでノートブックを実行してください。')
# PRのレビュー中も実行可能にし、merge後は自動的にmainを使います。
if 'google.colab' in sys.modules and not (PROJECT_ROOT / 'src/chemistory_gpr/kernels.py').exists():
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', '--depth', '1', 'origin', FALLBACK_REF], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
os.chdir(PROJECT_ROOT)
subprocess.run([sys.executable, 'scripts/prepare_data.py'], check=True)
# このセルだけで、後続セルのchemistory_gpr importまで準備します。
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_ROOT)], check=True)
src_dir = str(PROJECT_ROOT / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
import importlib
importlib.invalidate_caches()
import chemistory_gpr
print('PROJECT_ROOT =', PROJECT_ROOT)
print('chemistory_gpr =', chemistory_gpr.__file__)

In [ ]:
# Python環境
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
RESULTS = PROJECT_ROOT / 'results'
RESULTS.mkdir(exist_ok=True)

## A. 入力データの整合性検査

In [ ]:
from chemistory_gpr.handoff import load_handoff_data

DATA_DIR = PROJECT_ROOT / 'data' / 'gpr_handoff'
data = load_handoff_data(DATA_DIR)
print('n =', len(data.y))
print('base feature count =', data.base.shape[1] - 2)
print('X_proc feature count =', data.xproc.shape[1] - 1)
print('fold counts =', pd.Series(data.fold_id).value_counts().sort_index().to_dict())
display(pd.read_csv(DATA_DIR / '04_reference_RF_results.csv'))

## B. R randomForestの再現

RFはRの `randomForest` と `pls::plsr(method='simpls')` をそのまま使います。R 3.6で `sample()` の方式が変わったため、現在の `Rejection` と旧 `Rounding` を両方試します。

In [ ]:
import shutil

IN_COLAB = 'google.colab' in sys.modules
if shutil.which('Rscript') is None and IN_COLAB:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'r-base', 'r-cran-randomforest', 'r-cran-pls'], check=True)
elif shutil.which('Rscript') is None:
    print('この環境にはRscriptがないためRFセルだけをスキップします。Colabでは自動導入されます。')
else:
    package_check = 'missing <- c("randomForest","pls")[!sapply(c("randomForest","pls"), requireNamespace, quietly=TRUE)]; if(length(missing)) install.packages(missing, repos="https://cloud.r-project.org")'
    subprocess.run(['Rscript', '-e', package_check], check=True)
print('Rscript =', shutil.which('Rscript'))

In [ ]:
if shutil.which('Rscript'):
    subprocess.run([sys.executable, 'scripts/run_rf_reproduction.py'], check=True)
    rf_metrics = pd.read_csv(RESULTS / 'rf_reproduction_all_metrics.csv')
    rf_comparison = pd.read_csv(RESULTS / 'rf_reproduction_comparison.csv')
    display(rf_metrics)
    display(rf_comparison)
else:
    print('RF再現は未実行です。')

## C. GPR候補の固定10-fold比較

周期角度 + `X_proc` fold内PCA8を共通にし、Matérn 1/2・3/2・5/2、等方RBF、Rational Quadratic、RBF-ARD、Linearを比較します。すべてWhiteKernel付きです。

既定では計算の重い120次元RBF-ARDだけコミット済み結果を読み、他6候補を再fitします。`INCLUDE_ARD=True` でARDも再計算できます。`ARD_RESTARTS=0` でも帯域幅は第二種最尤法で最適化されます。`R2` は相関係数の二乗ではなく、`1-SSE/SST` です。

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
from chemistory_gpr.handoff import cross_validate_handoff, handoff_kernel_candidates

INCLUDE_ARD = False  # TrueではColabで数分以上かかることがあります
ARD_RESTARTS = 0
committed_metrics = pd.read_csv(RESULTS / 'gpr_handoff_metrics.csv')
candidate_metrics = []
candidate_predictions = {}
configs = handoff_kernel_candidates(rbf_ard_restarts=ARD_RESTARTS)
if not INCLUDE_ARD:
    configs = [config for config in configs if not config.ard]
warnings.filterwarnings('ignore', category=ConvergenceWarning)
for config in configs:
    print('Running', config.name)
    prediction, metrics = cross_validate_handoff(data, config)
    candidate_predictions[config.name] = prediction
    metrics = {k: v for k, v in metrics.items() if k not in {'kernels','kernel_diagnostics'}}
    candidate_metrics.append(metrics)
    prediction.to_csv(RESULTS / f'gpr_handoff_oof_{config.name}.csv', index=False)
metric_table = pd.DataFrame(candidate_metrics)
if not INCLUDE_ARD:
    ard_reference = committed_metrics.loc[committed_metrics['ard'].astype(bool)].copy()
    metric_table = pd.concat([metric_table, ard_reference], ignore_index=True)
    for name in ard_reference['model']:
        candidate_predictions[name] = pd.read_csv(RESULTS / f'gpr_handoff_oof_{name}.csv')
metric_table = metric_table.sort_values('R2', ascending=False)
metric_table.to_csv(RESULTS / 'gpr_handoff_metrics.csv', index=False)
metric_table['upper_bound_fraction'] = np.where(
    metric_table['length_scales_total'] > 0,
    metric_table['length_scales_at_upper_bound_total'] / metric_table['length_scales_total'],
    np.nan,
)
display(metric_table[['model','kernel_family','ard','optimizer_restarts','R2','corr2','RMSE','MAE','coverage_95','NLPD','upper_bound_fraction']])

In [ ]:
# RFを主基準に加え、fold別・試料別の挙動診断を作成
from chemistory_gpr.handoff_report import build_handoff_report

report_paths = build_handoff_report(DATA_DIR, RESULTS)
primary = pd.read_csv(report_paths['comparison'])
behavior = pd.read_csv(report_paths['behavior_summary'])
fold_vs_rf = pd.read_csv(report_paths['best_vs_rf_fold_metrics'])
largest_errors = pd.read_csv(report_paths['largest_errors'])
display(primary[['rank_R2','source','model','kernel_family','R2','RMSE','MAE','coverage_95','NLPD','fold_RMSE_wins_out_of_10']])
display(behavior)
display(largest_errors[['file_key','fold','y','pred_mean','pred_std','rf_pred','gpr_abs_error','rf_abs_error']].head(10))

In [ ]:
# 最良GPRとRFのOOF予測・fold別RMSE・不確実性の挙動
best_name = metric_table.iloc[0]['model']
best = candidate_predictions[best_name]
paired = pd.read_csv(report_paths['paired_predictions'])
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].scatter(paired['y'], paired['rf_pred'], s=18, alpha=.55, label='RF')
axes[0].scatter(paired['y'], paired['pred_mean'], s=18, alpha=.55, label='best GPR')
limits = [paired['y'].min(), paired['y'].max()]
axes[0].plot(limits, limits, '--', color='black')
axes[0].set(xlabel='Observed y', ylabel='OOF prediction', title='RF vs best GPR')
axes[0].legend()
x = np.arange(len(fold_vs_rf))
axes[1].bar(x-.2, fold_vs_rf['RF_RMSE'], width=.4, label='RF')
axes[1].bar(x+.2, fold_vs_rf['RMSE'], width=.4, label='best GPR')
axes[1].set(xticks=x, xticklabels=fold_vs_rf['fold'], xlabel='fold', ylabel='RMSE', title='Performance is fold-dependent')
axes[1].legend()
axes[2].scatter(paired['pred_std'], paired['gpr_abs_error'], s=20, alpha=.65)
rho = paired['pred_std'].corr(paired['gpr_abs_error'], method='spearman')
axes[2].set(xlabel='GPR predictive std', ylabel='absolute error', title=f'Uncertainty ranking: Spearman={rho:.3f}')
plt.tight_layout(); plt.show()

### 読み方

Matérn 3/2は受領RF報告値よりR²が約0.026高く、RMSEを約15%減らします。一方、等方RBF・Rational Quadratic・Matérn 5/2との差は非常に小さく、foldごとの首位も入れ替わります。Linearの大幅低下は非線形性の必要性を、RBF-ARDの低下と長さ尺度上限到達は過剰パラメータ化を示します。最良GPRでもRFに負けるfoldと大誤差例があり、予測標準偏差と絶対誤差の対応も強くありません。同じCVで候補を選んでいるため、最終確定にはnested CVまたは独立データが必要です。